# LAB 04 - Experiment 4: Word order (Mục 17)
Minh họa giới hạn của Average Word2Vec với hai câu:

- S1: `this movie is good`
- S2: `this movie is not good`

Notebook dùng cùng IMDB, chia 60/20/20 như các mục trước, huấn luyện Word2Vec chỉ trên train. Sau đó tính average embedding, cosine similarity và so sánh S2 với câu đảo thứ tự `this movie is good not`.

In [2]:
# Nếu Colab chưa có gensim, cài thư viện này.
try:
    import gensim
except ImportError:
    %pip -q install gensim

import scipy.linalg, numpy as np
# Tương thích với một số bản SciPy đã bỏ scipy.linalg.triu
if not hasattr(scipy.linalg, 'triu'):
    scipy.linalg.triu = np.triu
from gensim.models import Word2Vec
print('gensim:', gensim.__version__)

gensim: 4.4.0


In [3]:
from pathlib import Path
import zipfile, re, html
import pandas as pd

ZIP_PATH = Path('/content/imdb.zip')
if not ZIP_PATH.exists():
    from google.colab import files
    print('Chọn imdb.zip để tải lên Colab.')
    uploaded = files.upload()
    ZIP_PATH = Path(next(iter(uploaded)))

def load_imdb_zip(zip_path):
    frames=[]
    with zipfile.ZipFile(zip_path) as z:
        csvs=[n for n in z.namelist() if n.lower().endswith('.csv')]
        if not csvs:
            raise ValueError('Không tìm thấy CSV trong ZIP.')
        for name in csvs:
            with z.open(name) as f:
                part=pd.read_csv(f)
            part.columns=[str(c).strip().lower() for c in part.columns]
            if {'review','sentiment'}.issubset(part.columns):
                part=part[['review','sentiment']].copy()
            elif {'text','label'}.issubset(part.columns):
                part=part.rename(columns={'text':'review','label':'sentiment'})[['review','sentiment']].copy()
            else:
                raise ValueError(f'{name}: cần cột review/sentiment hoặc text/label.')
            frames.append(part)
    return pd.concat(frames,ignore_index=True)

df=load_imdb_zip(ZIP_PATH)
df['review']=df['review'].fillna('').astype(str)
df['sentiment']=df['sentiment'].astype(str).str.strip().str.lower()
df=df[df['sentiment'].isin(['positive','negative'])].drop_duplicates('review').reset_index(drop=True)
df['label']=df['sentiment'].map({'negative':0,'positive':1})

def normalize_review(text):
    text=html.unescape(text)
    text=re.sub(r'<[^>]+>',' ',text)
    return re.sub(r'\s+',' ',text).strip().lower()

df['text_clean']=df['review'].map(normalize_review)
print('Review sau loại trùng:',f'{len(df):,}')
print(df['sentiment'].value_counts().to_string())

Chọn imdb.zip để tải lên Colab.


Saving imdb.zip to imdb.zip
Review sau loại trùng: 49,582
sentiment
positive    24884
negative    24698


## Chia dữ liệu và huấn luyện Word2Vec
Chia phân tầng 60/20/20 với `random_state=42`. Word2Vec chỉ được fit trên review train. Huấn luyện trên toàn bộ train có thể mất vài phút trong Colab.

In [4]:
from sklearn.model_selection import train_test_split
import time

X_trainval,X_test,y_trainval,y_test=train_test_split(
    df['text_clean'],df['label'],test_size=0.20,random_state=42,stratify=df['label']
)
X_train,X_val,y_train,y_val=train_test_split(
    X_trainval,y_trainval,test_size=0.25,random_state=42,stratify=y_trainval
)

def tokenize(text):
    return re.findall(r'\b\w+\b',text.lower())

train_sentences=[tokenize(text) for text in X_train]
t0=time.perf_counter()
w2v=Word2Vec(
    sentences=train_sentences,vector_size=100,window=5,min_count=2,
    workers=2,sg=1,epochs=5,seed=42
)
print(f'Train reviews: {len(X_train):,}')
print(f'Word2Vec vocabulary: {len(w2v.wv):,}')
print(f'Word2Vec training time: {time.perf_counter()-t0:.2f} seconds')

Train reviews: 29,748
Word2Vec vocabulary: 51,395
Word2Vec training time: 191.27 seconds


## Tạo average embedding và so sánh câu
Average embedding: \(\bar{v}=\frac{1}{n}\sum_i v_i\). Cosine similarity gần 1 nghĩa là hai vector cùng hướng; tuy nhiên, độ giống vector cao không đảm bảo cùng polarity.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def average_embedding(text, model):
    tokens=tokenize(text)
    known=[model.wv[token] for token in tokens if token in model.wv]
    if not known:
        return np.zeros(model.vector_size,dtype=np.float32),tokens,[]
    return np.mean(known,axis=0),tokens,[t for t in tokens if t in model.wv]

def cosine(a,b):
    return float(cosine_similarity(a.reshape(1,-1),b.reshape(1,-1))[0,0])

S1='this movie is good'
S2='this movie is not good'
S2_reordered='this movie is good not'

v1,t1,k1=average_embedding(S1,w2v)
v2,t2,k2=average_embedding(S2,w2v)
v2r,t2r,k2r=average_embedding(S2_reordered,w2v)

print('S1 tokens:',t1)
print('S2 tokens:',t2)
print('S1 in-vocabulary:',k1)
print('S2 in-vocabulary:',k2)
print(f'Cosine(S1, S2): {cosine(v1,v2):.4f}')
print(f'Cosine(S2, reordered S2): {cosine(v2,v2r):.4f}')
print(f'L2 distance(S1, S2): {np.linalg.norm(v1-v2):.4f}')
print('Word "not" has a learned vector:', 'not' in w2v.wv)
if 'not' in w2v.wv:
    print(f'Norm of not vector: {np.linalg.norm(w2v.wv["not"]):.4f}')

comparison=pd.DataFrame({
    'Sentence':[S1,S2,S2_reordered],
    'Tokens':[t1,t2,t2r],
    'Cosine to S1':[1.0,cosine(v1,v2),cosine(v1,v2r)]
})
display(comparison)

S1 tokens: ['this', 'movie', 'is', 'good']
S2 tokens: ['this', 'movie', 'is', 'not', 'good']
S1 in-vocabulary: ['this', 'movie', 'is', 'good']
S2 in-vocabulary: ['this', 'movie', 'is', 'not', 'good']
Cosine(S1, S2): 0.9795
Cosine(S2, reordered S2): 1.0000
L2 distance(S1, S2): 0.4143
Word "not" has a learned vector: True
Norm of not vector: 2.6705


,Sentence,Tokens,Cosine to S1
0,this movie is good,"[this, movie, is, good]",1.000000
1,this movie is not good,"[this, movie, is, not, good]",0.979534
2,this movie is good not,"[this, movie, is, good, not]",0.979534


## Trả lời câu hỏi mục 17
Trong average word embeddings, `not` không bị bỏ đi: nếu có trong vocabulary, vector của nó được cộng cùng các vector từ còn lại. Tuy nhiên mô hình chỉ lấy trung bình, không có phép toán phủ định và không giữ vị trí/tác động của `not` lên `good`. Vì vậy hai câu có nhiều từ chung có thể có vector gần nhau dù sentiment trái ngược.

Phép thử S2 và `this movie is good not` cho thấy đổi thứ tự không đổi tập từ, nên average embedding của chúng giống nhau (cosine xấp xỉ 1). Điều này chứng minh biểu diễn trung bình bag-of-words không mã hóa word order. Xem cosine thực tế ở output để thấy mức tương tự giữa S1 và S2 với bộ embedding đã học.